In [1]:
import numpy as np
import pandas as pd

In [3]:
# copied from as2.py
OPCODES = {
    "nop": 0,
    "+": 1,
    "-": 2,
    "*": 3,
    "/": 4,
    "%": 5,
    ">>": 6,
    "<<": 7,
    "~": 8,
    ">": 9,
    "<": 10,
    ">=": 11,
    "<=": 12,
    "==": 13,
    "!=": 14,
    "jump": 15,
    "jump_imm": 16,
    "call": 17,  # probably don't need this one
    "call_imm": 18,
    "btrue": 19,
    "btrue_imm": 20,
    "bfalse": 21,
    "bfalse_imm": 22,
    "return": 23,
    ";": 23,  # alias
    "load_module": 24,
    "extern_call": 25,
    "loadword": 26,
    "@": 26,  # alias
    "storeword": 27,
    "!": 27,  # alias
    "push_imm": 28,
    "dup": 29,
    "swap": 30,
    "drop": 31,
    "over": 32,
    "rot": 33,
    "dup2": 34,
    "swap2": 35,
    "over2": 36,
    "drop2": 37,  # why?
    "rpush": 38,
    ">r": 38,  # alias
    "rpop": 39,
    "r>": 39,  # alias
    "rcopy": 40,
    "r@": 40,  # alias
    "inc": 41,
    "dec": 42,
    "rcopy2": 43,
    "loadbyte": 44,
    "@b": 44,
    "storebyte": 45,
    "!b": 45,
    "pick": 46,
    "and": 47,
    "or": 48,
    "xor": 49,
    "lsr": 50,
}

# invert the dict
ops = {}
for k, v in OPCODES.items():
    ops[v] = k

In [4]:
df = pd.read_csv("running_opcodes.csv", names=["opcode"])
df["opcode_name"] = df.apply(lambda r: ops[int(r["opcode"])], axis=1)

In [12]:
df

,opcode,opcode_name
0,18,call_imm
1,28,push_imm
2,26,@
3,28,push_imm
4,25,extern_call
...,...,...
26859,26,@
26860,28,push_imm
26861,25,extern_call
26862,23,;


In [6]:
# create a new df where the columns are values in a rolling window of winsize
def rolling_to_df(df, winsize):
    rol = df.rolling(winsize)
    rows = []
    for window in rol:
        ops = window.reset_index(drop=True)["opcode_name"]
        rows.append(ops)
    return pd.DataFrame(rows)

In [7]:
def markov_count(df, winsize):
    win = rolling_to_df(df, winsize)
    return win.groupby(list(range(winsize)), as_index=False).value_counts().sort_values(by="count", ascending=False)

In [10]:
pd.set_option('display.max_rows', 50)

In [11]:
markov_count(df, 5)

,0,1,2,3,4,count
113,call_imm,call_imm,push_imm,;,*,1057
122,call_imm,push_imm,;,*,push_imm,876
183,push_imm,;,*,push_imm,+,876
23,+,push_imm,+,;,@,800
177,push_imm,+,push_imm,+,;,800
...,...,...,...,...,...,...
132,call_imm,push_imm,push_imm,call_imm,swap,1
131,call_imm,push_imm,call_imm,push_imm,@,1
128,call_imm,push_imm,@,push_imm,@,1
127,call_imm,push_imm,@,push_imm,/,1


# unused opcodes

In [29]:
df_comptime = pd.read_csv("opcodes.csv", names=["opcode"])

In [30]:
opcodes_deduped = {}
for op, val in OPCODES.items():
    inthere = False
    for ddval in opcodes_deduped.keys():
        if ddval == val:
            inthere = True
            break
    if not inthere:
        opcodes_deduped[val] = op

print(opcodes_deduped)

{0: 'nop', 1: '+', 2: '-', 3: '*', 4: '/', 5: '%', 6: '>>', 7: '<<', 8: '~', 9: '>', 10: '<', 11: '>=', 12: '<=', 13: '==', 14: '!=', 15: 'jump', 16: 'jump_imm', 17: 'call', 18: 'call_imm', 19: 'btrue', 20: 'btrue_imm', 21: 'bfalse', 22: 'bfalse_imm', 23: 'return', 24: 'load_module', 25: 'extern_call', 26: 'loadword', 27: 'storeword', 28: 'push_imm', 29: 'dup', 30: 'swap', 31: 'drop', 32: 'over', 33: 'rot', 34: 'dup2', 35: 'swap2', 36: 'over2', 37: 'drop2', 38: 'rpush', 39: 'rpop', 40: 'rcopy', 41: 'inc', 42: 'dec', 43: 'rcopy2', 44: 'loadbyte', 45: 'storebyte', 46: 'pick', 47: 'and', 48: 'or', 49: 'xor', 50: 'lsr'}


In [31]:
for opval, opname in opcodes_deduped.items():
    if opval not in df_comptime["opcode"].to_numpy():
        print(opname)

nop
>>
~
<=
jump
call
btrue
btrue_imm
bfalse
dup2
swap2
over2
drop2
pick
